In [16]:
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

In [17]:
# ==========================================
# 1. CARREGAMENTO DOS DADOS (ARQUIVOS LOCAIS)
# ==========================================
# Replace these with the exact names of the files you just uploaded
file_true = '/test1-rotulado-para-teste-estatistico.xlsx' # Note: df_true was originally an Excel sheet
file_ep1 = '/test1modelo1.csv'
file_ep2 = '/test1modelo2.csv'

# Loads the ground truth (handles both Excel and CSV formats automatically)
if file_true.endswith(('.xls', '.xlsx')):
    df_true = pd.read_excel(file_true)
else:
    df_true = pd.read_csv(file_true)

df_ep1 = pd.read_csv(file_ep1)
df_ep2 = pd.read_csv(file_ep2)

# ==========================================
# ATENÇÃO: Extraindo os rótulos da SEGUNDA coluna (índice 1)
# ==========================================
y_true = df_true.iloc[:, 1].values
y_ep1 = df_ep1.iloc[:, 1].values
y_ep2 = df_ep2.iloc[:, 1].values

In [18]:
# ==========================================
# 2. CÁLCULO DE ACERTOS E ERROS
# ==========================================
# Cria arrays booleanos (True para acerto, False para erro)
acertos_ep1 = (y_ep1 == y_true)
acertos_ep2 = (y_ep2 == y_true)

In [19]:
# ==========================================
# 3. TABELA DE CONTINGÊNCIA
# ==========================================
# a: Ambos erraram
# b: EP1 errou, Atual acertou
# c: EP1 acertou, Atual errou
# d: Ambos acertaram
a = np.sum((~acertos_ep1) & (~acertos_ep2))
b = np.sum((~acertos_ep1) & (acertos_ep2))
c = np.sum((acertos_ep1) & (~acertos_ep2))
d = np.sum((acertos_ep1) & (acertos_ep2))

tabela = [[a, b],
          [c, d]]

In [20]:
# ==========================================
# 4. TESTE DE MCNEMAR
# ==========================================
# Usamos exact=False para usar a distribuição Qui-quadrado (recomendado pois b+c > 25)
# correction=True aplica a correção de continuidade de Edwards
resultado_mcnemar = mcnemar(tabela, exact=False, correction=True)
p_valor = resultado_mcnemar.pvalue
estatistica = resultado_mcnemar.statistic

# Nível de significância (alfa)
alpha = 0.05

In [21]:
# ==========================================
# 5. GERAÇÃO DO RELATÓRIO
# ==========================================
relatorio = f"""
=========================================================
          RELATÓRIO DE COMPARAÇÃO - TESTE DE MCNEMAR
=========================================================
Total de instâncias avaliadas: {len(y_true)}

TABELA DE CONTINGÊNCIA:
---------------------------------------------------------
                      | Modelo Atual Acertou | Modelo Atual Errou
---------------------------------------------------------
Modelo EP1 Acertou    | {d:^20} | {c:^18}
Modelo EP1 Errou      | {b:^20} | {a:^18}
---------------------------------------------------------

RESULTADOS ESTATÍSTICOS:
---------------------------------------------------------
Estatística Qui-quadrado: {estatistica:.4f}
P-valor                 : {p_valor:.4e}
Nível de Significância  : {alpha}

CONCLUSÃO:
---------------------------------------------------------
"""

if p_valor < alpha:
    relatorio += (
        "Rejeita-se a hipótese nula (p < 0.05).\n"
        "Existe uma diferença estatisticamente significativa no desempenho \n"
        "entre o modelo entregue no EP1 e o modelo atual."
    )
else:
    relatorio += (
        "Falha ao rejeitar a hipótese nula (p >= 0.05).\n"
        "NÃO existe uma diferença estatisticamente significativa no desempenho \n"
        "entre o modelo entregue no EP1 e o modelo atual."
    )

print(relatorio)

# Opcional: Salvar o relatório em um arquivo de texto
with open('relatorio_mcnemar.txt', 'w', encoding='utf-8') as f:
    f.write(relatorio)
print("\n[!] Relatório salvo como 'relatorio_mcnemar.txt'")


          RELATÓRIO DE COMPARAÇÃO - TESTE DE MCNEMAR
Total de instâncias avaliadas: 900

TABELA DE CONTINGÊNCIA:
---------------------------------------------------------
                      | Modelo Atual Acertou | Modelo Atual Errou
---------------------------------------------------------
Modelo EP1 Acertou    |         365          |         82        
Modelo EP1 Errou      |          96          |        357        
---------------------------------------------------------

RESULTADOS ESTATÍSTICOS:
---------------------------------------------------------
Estatística Qui-quadrado: 0.9494
P-valor                 : 3.2986e-01
Nível de Significância  : 0.05

CONCLUSÃO:
---------------------------------------------------------
Falha ao rejeitar a hipótese nula (p >= 0.05).
NÃO existe uma diferença estatisticamente significativa no desempenho 
entre o modelo entregue no EP1 e o modelo atual.

[!] Relatório salvo como 'relatorio_mcnemar.txt'
